# Reading Races data from Bronze layer table

In [0]:
%run ../00-common/01.environment_config

In [0]:
%run ../00-common/03.Silver_Helper

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.races"
sliver_table = f"{catalog_name}.{silver_schema}.races"

In [0]:
from pyspark.sql.functions import*

In [0]:
races_df= spark.read.table(bronze_table).filter(col('batch_id') == v_batch_id)

# select only required columns

In [0]:
races_select_df = races_df.select("season", "round", "circuitId", "raceName", "date", "ingestion_timestamp", "source_files",col("batch_id"))


In [0]:
races_renamed_df = (races_select_df
                    .withColumnsRenamed({"circuitId":"circuit_id", "raceName":"race_name", "date":"race_date"})

                    )

In [0]:
races_distincts_df = races_renamed_df.dropDuplicates(["season",'round'])


In [0]:
races_final_df = (races_distincts_df
                .withColumn('race_name',initcap(col("race_name")))
)


In [0]:
columns_to_update=[
        "race_name",
        "race_date",
        "circuit_id",
        "ingestion_timestamp",
        "source_files",
        "batch_id"
    ]

In [0]:
write_to_silver(input_df=races_final_df,target_table=sliver_table,
                merge_condn="t.season=s.season AND t.round = s.round",
                columns_to_update=columns_to_update
                )

In [0]:
display(spark.table(sliver_table))